# PDF Extraction Pipeline

Set PDF_PATH in the code cell and run it. It creates a layout PDF and a text file containing table data, image bounding boxes, and text blocks.


In [ ]:
!pip install -q pymupdf pdfplumber pdf2image pillow matplotlib pandas
!pip install -q "camelot-py[cv]"
!pip install -q opencv-python-headless


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.2/113.2 kB 5.3 MB/s eta 0:00:00


In [ ]:
import os
import pymupdf
import camelot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.backends.backend_pdf import PdfPages
from collections import defaultdict

In [ ]:
def overlap_ratio(candidate, reference):
    intersection = candidate & reference
    if intersection.is_empty:
        return 0

    candidate_area = candidate.width * candidate.height
    if candidate_area == 0:
        return 0
    overlap_area = intersection.width * intersection.height
    ratio = overlap_area / candidate_area
    return ratio


def overlaps_fraud(candidate, threshold=0.50):
    for fraud_rect in fraud_rects:
        ratio = overlap_ratio(candidate, fraud_rect)
        if ratio > threshold:
            return True
    return False


def matches_logo(candidate, threshold=0.50):
    for logo_rect in logo_rects:
        ratio = overlap_ratio(candidate, logo_rect)
        if ratio > threshold:
            return True
    return False


def overlaps_content(candidate, boxes, threshold=0.50):
    for content_rect in boxes:
        ratio = overlap_ratio(candidate, content_rect)
        if ratio > threshold:
            return True
    return False


def overlaps_header_footer(candidate):
    for zone in header_footer_zones:
        ratio = overlap_ratio(candidate, zone)
        if ratio > 0:
            return True
    return False

In [ ]:
def duplicates(PDF_PATH):
    doc = pymupdf.open(PDF_PATH)
    n = len(doc)

    # Select pages for duplicate detection
    if n < 10:
        page_count = min(n, max(2, n // 2)) if n >= 2 else n
    elif n <= 14:
        page_count = 5
    elif n <= 20:
        page_count = 7
    else:
        page_count = 10

    page_counts = list(range(page_count))

    # Collect bounding boxes and their occurrences
    bbox_occurrences = defaultdict(list)

    for page_num in page_counts:
        page = doc[page_num]

        # Collect text and image blocks
        for block in page.get_text("dict")["blocks"]:
            bbox = tuple(float(v) for v in block["bbox"])
            bbox_occurrences[bbox].append({
                "page": page_num + 1,
                "type": "text" if block.get("type") == 0 else "image"
            })

        # Collect drawing bounding boxes
        for drawing in page.get_drawings():
            rect = drawing.get("rect")
            if rect:
                bbox = tuple(float(v) for v in rect)
                bbox_occurrences[bbox].append({
                    "page": page_num + 1,
                    "type": "drawing"
                })

    fraud_box = []
    logo_box = []

    # Classify recurring boxes
    for bbox, occurrences in bbox_occurrences.items():
        pages = sorted(set(item["page"] for item in occurrences))
        # Keep boxes occurring on at least two pages
        if len(pages) < 2:
            continue

        x0, y0, x1, y1 = bbox
        bbox_area = max(0, x1 - x0) * max(0, y1 - y0)
        large_pages = set()

        for page_num in pages:
            page_area = doc[page_num - 1].rect.width * doc[page_num - 1].rect.height
            if page_area > 0 and bbox_area / page_area > 0.10:
                large_pages.add(page_num)

        is_image = any(item["type"] == "image" for item in occurrences)

        # Large recurring images are classified as logo candidates
        if is_image and len(large_pages) >= 5:
            logo_box.append(bbox)
        else:
            fraud_box.append(bbox)

    doc.close()

    return fraud_box, logo_box


In [ ]:
def header_footer(PDF_PATH, page_number, fraud_boxes):
    HEADER_FOOTER_MARGIN = 5
    doc = pymupdf.open(PDF_PATH)

    try:
        page = doc[page_number - 1]
        page_width = page.rect.width
        page_height = page.rect.height

        header_candidates = []
        footer_candidates = []

        for bbox in fraud_boxes:
            x0, y0, x1, y1 = bbox

            # Header candidates: top 2%
            if y0 < page_height * 0.02:
                header_candidates.append(bbox)

            # Footer candidates: bottom 2%
            if y1 > page_height * 0.98:
                footer_candidates.append(bbox)

        header_zone = None
        footer_zone = None

        # Header zone
        if header_candidates:
            header_bottom = max(bbox[3] for bbox in header_candidates)
            header_zone = pymupdf.Rect(
                0,
                0,
                page_width,
                min(page_height,header_bottom + HEADER_FOOTER_MARGIN)
          )

        # Footer zone
        if footer_candidates:
            footer_top = min(bbox[1] for bbox in footer_candidates)
            footer_zone = pymupdf.Rect(
                0,
                max(0, footer_top - HEADER_FOOTER_MARGIN),
                page_width,
                page_height
            )

        # Extend footer exclusion to the bottom 5%
        footer_zone_5pct = pymupdf.Rect(
            0,
            page_height * 0.95,
            page_width,
            page_height
        )

        if footer_zone is None:
            footer_zone = footer_zone_5pct
        else:
            footer_zone |= footer_zone_5pct

        header_footer_zones = [
            zone
            for zone in [header_zone, footer_zone]
            if zone is not None
        ]

        return header_footer_zones

    finally:
        doc.close()

In [ ]:
def extract_tables(PDF_PATH, PAGE_NUMBER, page, page_height, overlaps_header_footer,overlaps_fraud):
    table_boxes = []

    try:
        pymupdf_tables = page.find_tables()
        pymupdf_has_tables = len(pymupdf_tables.tables) > 0
    except Exception as e:
        pymupdf_has_tables = False

    if pymupdf_has_tables:
        try:
            tables = camelot.read_pdf(
                PDF_PATH,
                pages=str(PAGE_NUMBER),
                flavor="lattice"
            )

            if len(tables) == 0:
                tables = camelot.read_pdf(
                    PDF_PATH,
                    pages=str(PAGE_NUMBER),
                    flavor="stream"
                )

            for table in tables:
                x0, y0, x1, y1 = table._bbox

                # Camelot bottom-left -> PyMuPDF top-left
                rect = pymupdf.Rect(
                    x0,
                    page_height - y1,
                    x1,
                    page_height - y0
                )

                # Exclude full-width header/footer regions
                if overlaps_header_footer(rect):
                    continue

                # Exclude unwanted recurring boxes
                if overlaps_fraud(rect):
                    continue

                table_boxes.append(rect)
        except Exception as e:
            has_tables = False

    else:
        has_tables = False

    return table_boxes

In [ ]:
def extract_images(page, overlaps_header_footer, overlaps_fraud, matches_logo):
    image_boxes = []
    try:
        blocks = page.get_text("dict")["blocks"]

        for block in blocks:
            if block["type"] != 1:      # if block is image skip it
                continue

            rect = pymupdf.Rect(block["bbox"])

            if overlaps_header_footer(rect):           # Exclude header/footer regions
                continue

            if overlaps_fraud(rect):                 # Exclude unwanted recurring regions
                continue

            if matches_logo(rect):        # Exclude recurring logo/watermark images
                continue

            image_boxes.append(rect)

    except Exception as e:
        has_blocks = False

    return image_boxes

In [ ]:
def extract_text(page, PAGE_NUMBER, overlaps_header_footer, overlaps_fraud, overlaps_content, table_boxes, image_boxes):
    text_boxes = []
    text_records = []

    try:
        blocks = page.get_text("dict")["blocks"]
        for block_index, block in enumerate(blocks, start=1):

            if block["type"] != 0:            # Process text blocks only
                continue

            rect = pymupdf.Rect(block["bbox"])
            block_text = "\n".join(
                "".join(
                    span["text"]
                    for span in line.get("spans", [])
                )
                for line in block.get("lines", [])
            ).strip()

            if not block_text:
                continue

            # Exclude full-width header/footer regions
            if overlaps_header_footer(rect):
                continue

            # Exclude unwanted recurring text
            if overlaps_fraud(rect):
                continue

            # Exclude text blocks substantially inside tables
            if overlaps_content(rect, table_boxes):
                continue

            # Exclude text blocks substantially inside images
            if overlaps_content(rect, image_boxes):
                continue

            # Do not filter text using logo_boxes.
            # Preserve genuine text beneath watermarks.

            text_boxes.append(rect)
            text_records.append({
                "text_id": f"P{PAGE_NUMBER}_TXT{block_index}",
                "page": PAGE_NUMBER,
                "bbox": tuple(rect),
                "text": block_text
            })

    except Exception as e:
        has_text = False

    return text_boxes, text_records

In [ ]:
PDF_PATH = "/content/Dsa_project_report.pdf"   # keep your pdf path

OUTPUT_LAYOUT_PDF = "/content/extraction_layout__DSA_paper.pdf"  # enter your output pdf name with path
OUTPUT_TEXT_FILE = "/content/extracted_paper_DSA.txt"            # enter your output txt name with path


# 2. Find recurring boxes in the PDF

fraud_boxes, logo_boxes = duplicates(PDF_PATH)

fraud_rects = [pymupdf.Rect(box) for box in fraud_boxes]
logo_rects = [pymupdf.Rect(box) for box in logo_boxes]


# 3. Process every page and save the results
all_text_records = []

with pymupdf.open(PDF_PATH) as doc, PdfPages(OUTPUT_LAYOUT_PDF) as pdf:
    for page_number, page in enumerate(doc, start=1):

        page_width = page.rect.width
        page_height = page.rect.height

        # Find header and footer zones for this page
        header_footer_zones = header_footer(
            PDF_PATH, page_number, fraud_boxes
        )

        # Detect tables and images
        table_boxes = extract_tables(
            PDF_PATH, page_number, page, page_height,
            overlaps_header_footer, overlaps_fraud
        )

        image_boxes = extract_images(
            page, overlaps_header_footer,
            overlaps_fraud, matches_logo
        )

        # Extract text blocks
        text_boxes = []
        text_records = []

        for block_index, block in enumerate(
            page.get_text("dict")["blocks"], start=1
        ):
            if block.get("type") != 0:
                continue

            rect = pymupdf.Rect(block["bbox"])

            block_text = "\n".join(
                "".join(
                    span.get("text", "")
                    for span in line.get("spans", [])
                )
                for line in block.get("lines", [])
            ).strip()

            if not block_text:
                continue

            if overlaps_header_footer(rect):
                continue

            if overlaps_fraud(rect):
                continue

            if overlaps_content(rect, table_boxes):
                continue

            if overlaps_content(rect, image_boxes):
                continue

            text_boxes.append(rect)

            text_records.append({
                "page": page_number,
                "bbox": tuple(rect),
                "text": block_text
            })

        all_text_records.extend(text_records)

        # Render the original page
        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(1.5, 1.5),
            alpha=False
        )

        image = np.frombuffer(
            pix.samples, dtype=np.uint8
        ).reshape(pix.height, pix.width, pix.n)

        sx = pix.width / page_width
        sy = pix.height / page_height

        fig, ax = plt.subplots(figsize=(12, 16))
        ax.imshow(image)

        # Header and footer: orange
        for zone in header_footer_zones:
            ax.add_patch(patches.Rectangle(
                (zone.x0 * sx, zone.y0 * sy),
                zone.width * sx, zone.height * sy,
                edgecolor="orange", facecolor="none",
                linewidth=2
            ))

        # Tables: red
        for i, rect in enumerate(table_boxes, start=1):
            ax.add_patch(patches.Rectangle(
                (rect.x0 * sx, rect.y0 * sy),
                rect.width * sx, rect.height * sy,
                edgecolor="red", facecolor="none",
                linewidth=2
            ))
            ax.text(
                rect.x0 * sx, rect.y0 * sy,
                f"Table {i}", color="white",
                bbox=dict(facecolor="red")
            )

        # Images: blue
        for i, rect in enumerate(image_boxes, start=1):
            ax.add_patch(patches.Rectangle(
                (rect.x0 * sx, rect.y0 * sy),
                rect.width * sx, rect.height * sy,
                edgecolor="blue", facecolor="none",
                linewidth=2
            ))
            ax.text(
                rect.x0 * sx, rect.y0 * sy,
                f"Image {i}", color="white",
                bbox=dict(facecolor="blue")
            )

                  # TEXT BLOCKS: GREEN WITH ORDER NUMBERS

        for i, rect in enumerate(text_boxes, start=1):

            ax.add_patch(
                patches.Rectangle(
                    (rect.x0 * sx, rect.y0 * sy),
                    rect.width * sx,
                    rect.height * sy,
                    linewidth=1.5,
                    edgecolor="lime",
                    facecolor="none"
                )
            )

            # Display text-block number
            ax.text(
                rect.x0 * sx,
                rect.y0 * sy,
                f"Text {i}",
                color="black",
                fontsize=8,
                bbox=dict(facecolor="lime", alpha=0.8)
            )
        ax.set_title(f"Page {page_number}: Extraction Layout")
        ax.axis("off")
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)


# 4. Save extracted text to a text file
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as file:
    for record in all_text_records:
        file.write(f"\nPage: {record['page']}\n")
        file.write(f"Bounding box: {record['bbox']}\n")
        file.write(record["text"] + "\n")
        file.write("-" * 60 + "\n")


# 5. Download both files in Google Colab
from google.colab import files
files.download(OUTPUT_LAYOUT_PDF)
files.download(OUTPUT_TEXT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>